> **Step 1 — Environment Setup (Cell 1)**

In [14]:
# Install dependencies
!pip install timm einops -q

import os, math, random, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Multi-GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPUs available: {torch.cuda.device_count()}")

Using device: cuda
GPUs available: 1


> **Step 2 — Dataset Loading (Cell 2)**

In [4]:
BATCH_SIZE = 64
IMAGE_SIZE = 224
DATA_ROOT = os.path.join(akash2sharma_tiny_imagenet_path, "tiny-imagenet-200")

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std =[0.229,0.224,0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 100000 | Val: 10000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


> **Step 3 — Patchification & Masking Utilities (Cell 3)**

In [19]:
def patchify(images, patch_size=16):
    """
    images: (B, 3, H, W)
    returns patches: (B, N, patch_size*patch_size*3)
    where N = (H/patch_size)*(W/patch_size)
    """
    B, C, H, W = images.shape
    assert H % patch_size == 0 and W % patch_size == 0
    h = H // patch_size  # 14
    w = W // patch_size  # 14
    # Reshape to patches
    x = images.reshape(B, C, h, patch_size, w, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)           # (B, h, w, C, ps, ps)
    x = x.reshape(B, h * w, C * patch_size * patch_size)  # (B, N, patch_dim)
    return x

def unpatchify(patches, patch_size=16, image_size=224):
    """
    patches: (B, N, patch_size*patch_size*3)
    returns images: (B, 3, H, W)
    """
    B, N, D = patches.shape
    h = w = image_size // patch_size
    C = 3
    x = patches.reshape(B, h, w, C, patch_size, patch_size)
    x = x.permute(0, 3, 1, 4, 2, 5)           # (B, C, h, ps, w, ps)
    x = x.reshape(B, C, h * patch_size, w * patch_size)
    return x

def random_masking(x, mask_ratio=0.75):
    """
    x: (B, N, D) — all patch tokens
    Returns:
        x_visible:   (B, n_visible, D)
        mask:        (B, N) — 1=masked, 0=visible
        ids_restore: (B, N) — indices to restore original order
    """
    B, N, D = x.shape
    n_keep = int(N * (1 - mask_ratio))   # 49 visible patches

    noise = torch.rand(B, N, device=x.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :n_keep]
    x_visible = torch.gather(x, dim=1,
                             index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

    mask = torch.ones(B, N, device=x.device)
    mask[:, :n_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)

    return x_visible, mask, ids_restore

**Step 4 — Transformer Building Blocks (Cell 4)**




In [20]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv   = nn.Linear(dim, dim * 3, bias=True)
        self.proj  = nn.Linear(dim, dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadSelfAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim    = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x